# 1. Context

This notebook analyzes OCR performance of Google Document AI Solution over synthetic generated PDF images across variois degradation levels

# 2. Imports

In [1]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
import jiwer

In [23]:
import plotly.express as px

In [2]:
import sys

In [3]:
notebook_path = Path()
sys.path.append(str(notebook_path.resolve().parent))

In [4]:
from src.viz.helper import display_box_plot

# 3. Utils

In [5]:
# get csv paths for all language
results_root = Path("../results/google")
results_langs = [x.name.lower() for x in results_root.glob("*") if x.is_dir()]

In [6]:
results_langs

['gujarati',
 'oriya',
 'punjabi',
 'malayalam',
 'hindi',
 'manipuri',
 'konkani',
 'urdu',
 'telugu',
 'sanskrit',
 'assamese',
 'kashmiri',
 'english',
 'kannada',
 'tamil',
 'sindhi',
 'marathi',
 'santali',
 'nepali',
 'bengali',
 'maithali']

In [7]:
writing_system_to_language = {'Devanagari': ['hindi','sanskrit','nepali','konkani', 'maithali', 'marathi'],
                              'tamil': ['tamil'],'telugu': ['telugu'],'Kannada': ['kannada'],'Malayalam': ['malayalam'],
                              'Bengali': ['bengali', 'assamese'],'Meetei-mayek': ['manipuri'],'Gujarati': ['gujarati'],
                              'Gurmukhi': ['punjabi'],'Odia': ['oriya'],'Arabic': ['kashmiri', 'sindhi', 'urdu'],
                              'Latin': ['english'],'Ol-chiki': ['santali']}

In [8]:
language_to_writing_system = {
    "marathi": ["Devanagari"], "hindi": ["Devanagari"], "sanskrit": ["Devanagari"],
    "tamil": ["tamil"], "telugu": ["telugu"], "kannada": ["Kannada"],"malayalam": ["Malayalam"],
    "bengali": ["Bengali"], "assamese": ["Bengali"],"manipuri": ["Meetei-mayek"],"nepali": ["Devanagari"],
    "gujarati": ["Gujarati"], "punjabi": ["Gurmukhi"], "konkani": ["Devanagari"],"oriya": ["Odia"],"kashmiri": ["Arabic"], 
    "sindhi": ["Arabic", "Devanagari"], "urdu": ["Arabic"],"english": ["Latin"], "santali": ["Ol-chiki"],
    "maithali": ["Devanagari"]
    }

In [9]:
writing_sys_dict = defaultdict(list)

for language_res in results_langs:
    script = language_to_writing_system.get(language_res)[0]
    writing_sys_dict[script].append(language_res)

In [10]:
script_language_result = pd.Series(writing_sys_dict).to_frame(name='languages')
script_language_result.index.name = 'script'

## 3.1. Results Available Across Script and Language Combination

| script       | languages                                                         |
|:-------------|:------------------------------------------------------------------|
| Gujarati     | ['gujarati']                                                      |
| Odia         | ['oriya']                                                         |
| Gurmukhi     | ['punjabi']                                                       |
| Malayalam    | ['malayalam']                                                     |
| Devanagari   | ['hindi', 'konkani', 'sanskrit', 'marathi', 'nepali', 'maithali'] |
| Meetei-mayek | ['manipuri']                                                      |
| Arabic       | ['urdu', 'kashmiri', 'sindhi']                                    |
| telugu       | ['telugu']                                                        |
| Bengali      | ['assamese', 'bengali']                                           |
| Latin        | ['english']                                                       |
| Kannada      | ['kannada']                                                       |
| tamil        | ['tamil']                                                         |

In [11]:
def get_language_results_path(script: str, script_language_result: pd.DataFrame) -> list[Path]:
    """Get list of results path for given script"""

    results_script_lang = script_language_result.loc[script].to_list()[0]
    results_script_lang_path = [results_root.joinpath(lang).joinpath('results.csv') for lang in results_script_lang]

    return results_script_lang_path

In [12]:
def get_script_results(script: str, script_lang_df: pd.DataFrame):
    """Get results for a given script. output contains results for languages in the script"""

    results_script_path = get_language_results_path(script, script_lang_df)

    results_list = []
    for result_path in results_script_path:
        lang = result_path.parent.name
        df_res = pd.read_csv(result_path)
        df_res['language'] = lang
        results_list.append(df_res)

    results_script = pd.concat(results_list)
    results_script['script'] = script
    return results_script
    

## 3.2. Getting All the results

In [13]:
script_results_avail = script_language_result.index
results_consolidated = []
for script in script_results_avail:
    results_script = get_script_results(script=script, script_lang_df=script_language_result)
    results_consolidated.append(results_script)
consolidated_df = pd.concat(results_consolidated)

In [14]:
agg_results = (consolidated_df.groupby(['script', 'language']).agg(CER_AVG_L0=('cer_l0', 'median'),
                                                    CER_AVG_L1=('cer_l1', 'median'),
                                                    CER_AVG_L2=('cer_l2', 'median'),
                                                    CER_AVG_L3=('cer_l3', 'median'),
                                                    WER_AVG_L0=('wer_l0', 'median'),
                                                    WER_AVG_L1=('wer_l1', 'median'),
                                                    WER_AVG_L2=('wer_l2', 'median'),
                                                    WER_AVG_L3=('wer_l3', 'median')
                                                    ).round(3))

In [15]:
agg_results.columns = agg_results.columns.str.upper()

In [17]:
agg_results.to_clipboard(index=True) 

In [18]:
col_ord = ['file_id', 'language', 'script','ground_truth', 'ocr_output_L_0', 'ocr_output_L_1',
       'ocr_output_L_2', 'ocr_output_L_3', 'cer_l0', 'cer_l1', 'cer_l2',
       'cer_l3', 'wer_l0', 'wer_l1', 'wer_l2', 'wer_l3' ]
consolidated_df = consolidated_df[col_ord]

## upper casing column names
consolidated_df.columns = consolidated_df.columns.str.upper()

In [19]:
consolidated_df.round(3).to_clipboard(index=False)

# 4. Visualising Aggregate Results

In [21]:
agg_results_ = agg_results.reset_index()
agg_results_.columns = agg_results_.columns.str.upper()

In [27]:
agg_results_

,SCRIPT,LANGUAGE,CER_AVG_L0,CER_AVG_L1,CER_AVG_L2,CER_AVG_L3,WER_AVG_L0,WER_AVG_L1,WER_AVG_L2,WER_AVG_L3
0,Arabic,kashmiri,0.163,0.172,0.192,0.201,0.571,0.564,0.613,0.598
1,Arabic,sindhi,0.128,0.134,0.155,0.163,0.415,0.394,0.431,0.407
2,Arabic,urdu,0.024,0.021,0.032,0.034,0.063,0.060,0.104,0.073
3,Bengali,assamese,0.020,0.024,0.031,0.032,0.102,0.106,0.113,0.122
4,Bengali,bengali,0.021,0.027,0.030,0.032,0.077,0.085,0.095,0.090
5,Devanagari,hindi,0.005,0.007,0.014,0.013,0.022,0.023,0.029,0.035
6,Devanagari,konkani,0.016,0.024,0.026,0.033,0.106,0.106,0.111,0.113
7,Devanagari,maithali,0.015,0.018,0.019,0.027,0.081,0.084,0.088,0.106
8,Devanagari,marathi,0.008,0.015,0.017,0.016,0.036,0.042,0.042,0.055
9,Devanagari,nepali,0.008,0.010,0.016,0.012,0.039,0.046,0.048,0.044


In [31]:
lang_not_supported = ['kashmiri', 'sindhi', 'konkani', 'maithali', 'santali', 'manipuri']

In [32]:
agg_results_supported = agg_results_.loc[~agg_results_['LANGUAGE'].isin(lang_not_supported)]

In [35]:
agg_results_supported.to_clipboard(index=False)

In [39]:
print(agg_results_supported.to_markdown(index=False))

| SCRIPT     | LANGUAGE   |   CER_AVG_L0 |   CER_AVG_L1 |   CER_AVG_L2 |   CER_AVG_L3 |   WER_AVG_L0 |   WER_AVG_L1 |   WER_AVG_L2 |   WER_AVG_L3 |
|:-----------|:-----------|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|-------------:|
| Arabic     | urdu       |        0.024 |        0.021 |        0.032 |        0.034 |        0.063 |        0.06  |        0.104 |        0.073 |
| Bengali    | assamese   |        0.02  |        0.024 |        0.031 |        0.032 |        0.102 |        0.106 |        0.113 |        0.122 |
| Bengali    | bengali    |        0.021 |        0.027 |        0.03  |        0.032 |        0.077 |        0.085 |        0.095 |        0.09  |
| Devanagari | hindi      |        0.005 |        0.007 |        0.014 |        0.013 |        0.022 |        0.023 |        0.029 |        0.035 |
| Devanagari | marathi    |        0.008 |        0.015 |        0.017 |        0.016 |        0.036 |        0.

# 4. Results Across Various Writing system (samples)

# 4.1. Devanagari 

In [ ]:
results_devanagari = get_script_results(script='Devanagari', script_lang_df=script_language_result)

## 4.1.1. Box Plot Visualisation

In [ ]:
display_box_plot(results_df=results_devanagari, script=script, metric_type='CER', height=1500, width=1600).show()

In [ ]:
display_box_plot(results_df=results_devanagari, script=script, metric_type='WER',height=1500, width=1600).show()

## 5.1. Bengali

In [ ]:
script = 'Bengali'
results_bengali = get_script_results(script='Bengali', script_lang_df=script_language_result, )

### 5.1.1. Box Plot Visualisation

In [ ]:
display_box_plot(results_df=results_bengali, script=script, metric_type='CER').show()

In [ ]:
display_box_plot(results_df=results_bengali, script=script, metric_type='WER').show()

# 6. Addendum

## 6.1. Assessing WERs

### 6.1.1. Kannada

In [ ]:
script = 'Kannada'
results_kn = get_script_results(script=script, script_lang_df=script_language_result)

In [ ]:
idx = 0
gt = results_kn.loc[idx]['ground_truth']
ocred = results_kn.loc[idx]['ocr_output_L_0']

In [ ]:
output_wer = jiwer.process_words(gt, ocred)

In [ ]:
print(jiwer.visualize_alignment(output_wer, line_width=20))

### 6.1.2. Maithali

In [ ]:
idx = 2
results_maithali = results_devanagari.loc[results_devanagari['language'] == 'maithali']

In [ ]:
idx = 2
gt = results_maithali.loc[idx]['ground_truth']
ocred = results_maithali.loc[idx]['ocr_output_L_0']

In [ ]:
output_wer = jiwer.process_words(gt, ocred)

In [ ]:
print(jiwer.visualize_alignment(output_wer, line_width=15))

### 6.1.3. Arabic Languages

In [ ]:
results_urdu = consolidated_df.loc[consolidated_df['LANGUAGE'] == 'urdu']
results_kashmiri = consolidated_df.loc[consolidated_df['LANGUAGE'] == 'kashmiri']
results_sindhi = consolidated_df.loc[consolidated_df['LANGUAGE'] == 'sindhi']

#### 6.1.3.1. High Accuracy in Urdu

In [ ]:
results_urdu.loc[results_urdu['FILE_ID'] == 'Arabic_Noto_Naskh_Arabic_1']

In [ ]:
idx = 18
gt = results_urdu.loc[idx]['GROUND_TRUTH']
ocred = results_urdu.loc[idx]['OCR_OUTPUT_L_0']

In [ ]:
output_wer = jiwer.process_words(gt, ocred)

In [ ]:
output_cer = jiwer.process_characters(gt, ocred)

In [ ]:
print(jiwer.visualize_alignment(output_wer, line_width=15))

In [ ]:
print(ocred)

In [ ]:
print(gt)

### 6.1.3.2. Low Accuracy in Sindhi

In [ ]:
idx = 0
gt_ks = results_sindhi.loc[idx]['GROUND_TRUTH']
ocred_ks = results_sindhi.loc[idx]['OCR_OUTPUT_L_0']

In [ ]:
print(gt_ks)

In [ ]:
print(ocred_ks)

In [ ]:
results_sindhi.loc[idx]

In [ ]:
results_sindhi.loc[idx]